In [0]:
dbutils.widgets.text("FileName", "")
file_name = dbutils.widgets.get("FileName")

silver_path = f"abfss://data@karansa2s.dfs.core.windows.net/silver/{file_name.replace('.csv','')}/"


In [0]:
from pyspark.sql.functions import col, sum as _sum, count, desc

storage_account = "karansa2s"
storage_key = "<KEY>"

spark.conf.set(
  f"fs.azure.account.key.{storage_account}.dfs.core.windows.net",
  storage_key
)

silver_path = "abfss://data@karansa2s.dfs.core.windows.net/silver/sales_2025-12-01/"

df_silver = spark.read.csv(silver_path)

display(df_silver)


#### GOLD TABLE 1
##### Daily Sales Summary

In [0]:
daily_sales = df_silver.groupBy("SaleDate").agg(
    _sum("Amount").alias("TotalRevenue"),
    _sum("Qty").alias("TotalQty"),
    count("*").alias("TransactionCount")
)

display(daily_sales)


#### GOLD TABLE 2
##### Sales by Store

In [0]:
sales_by_store = df_silver.groupBy("Store").agg(
    _sum("Amount").alias("TotalRevenue"),
    _sum("Qty").alias("TotalQty"),
    count("*").alias("TransactionCount")
)

display(sales_by_store)


#### GOLD TABLE 3
##### Top Selling Items

In [0]:
top_items = df_silver.groupBy("Item").agg(
    _sum("Qty").alias("TotalQtySold")
).orderBy(desc("TotalQtySold"))

display(top_items)


##### WRITE GOLD OUTPUTS

In [0]:
daily_sales.write.mode("overwrite").csv("abfss://data@karansa2s.dfs.core.windows.net/gold/daily_sales")
sales_by_store.write.mode("overwrite").csv("abfss://data@karansa2s.dfs.core.windows.net/gold/sales_by_store")
top_items.write.mode("overwrite").csv("abfss://data@karansa2s.dfs.core.windows.net/gold/top_items")
